# 实验 01 · 从 GPU 到 verl：动手入门

本课使用 verl 0.7.1，参考 GPU 为 RTX 4060 8GB，无需下载模型。

先检查 GPU 是否可用，再比较 CPU 与 GPU 的计算时间，最后训练一个只有四个选项的小策略，观察奖励如何改变它的选择。预计 15 至 25 分钟；实际计算通常只需几分钟。

这是 JupyterLab 网页中的 Notebook。点击代码格，再按 **Shift + Enter**：运行当前格并前进。**Ctrl + Enter**：运行并停留。左侧 `[*]` 表示运行中，数字表示已执行。上方 ■ 可中断当前计算。

请从上到下运行。修改代码后可以重跑；要清空内存状态，选择 **Kernel → Restart Kernel and Run All Cells**。保存用 **Ctrl + S**。公开版已清空运行输出。执行代码后，页面会显示本次运行的结果；保存后再次打开时，请注意区分历史输出与新结果。

右上角内核请选择 **Python (verl GPU)**。GPU 在本机 WSL 中计算，网页负责显示结果。

## 0 · 准备工具

下一格只导入工具。Notebook 会保留变量，因此后面的格子可以使用这里的导入。

In [ ]:
import sys, time, statistics
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def emit(**values):
    print(values)

print('Python:', sys.executable)
assert sys.executable.startswith('/opt/verl-venv/'), '请选择 Python (verl GPU) 内核'
print('准备完成。接下来查看真实显卡。')

## 1 · 确认代码真的在 GPU 上运行

CUDA 是 PyTorch 使用 NVIDIA GPU 的计算接口。`device="cuda"` 才会把张量放到 GPU 上；安装好 GPU 版本不意味着所有代码会自动加速。

**观察：** 核对输出中的 GPU 型号和 CUDA 版本，并查看 FlashAttention 检查是否显示 PASS。GPU 型号应与你的设备一致。FlashAttention 是注意力运算的优化实现。

In [ ]:
import importlib.metadata as m
import torch
import verl
for package in ('verl', 'torch', 'vllm', 'flash-attn'):
    print(f'{package}: {m.version(package)}', flush=True)
print(f'Python: {sys.executable}\nverl: {verl.__file__}', flush=True)
assert torch.cuda.is_available(), 'CUDA 不可用，请通过 start-notebook.ps1 启动 WSL 环境'
print(f'GPU: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}', flush=True)
from flash_attn import flash_attn_func
q = torch.randn(1, 32, 2, 64, device='cuda', dtype=torch.float16)
assert torch.isfinite(flash_attn_func(q, q, q)).all()
print('PASS：真实 CUDA 运算与 FlashAttention 均通过。', flush=True)

## 2 · 测一次 CPU / GPU 加速

矩阵乘法是神经网络的重要基础操作。这里让 CPU 和 GPU 计算同一组 FP32 矩阵，先预热，再测 10 次，比较中位数。

GPU 运算通常异步提交，因此计时前后需要同步。这个实验**不含 CPU↔GPU 数据传输**，也不是完整训练的速度测试。后台负载、温度和电源模式会影响结果。

**动手：** 运行后把矩阵尺寸从 2048 改成 512 再试。小任务中 GPU 的启动开销可能抵消优势。

In [ ]:
import torch
torch.manual_seed(42)
a, b = (torch.randn(2048, 2048), torch.randn(2048, 2048))
times = {}
reference = None
print('FP32 / 2048×2048 / 预热 3 次 / 测量 10 次 / 不计数据传输', flush=True)
for device in ('cpu', 'cuda'):
    x, y = (a.to(device), b.to(device))
    for _ in range(3):
        z = x @ y
    if device == 'cuda':
        torch.cuda.synchronize()
    samples = []
    for i in range(10):
        start = time.perf_counter()
        z = x @ y
        if device == 'cuda':
            torch.cuda.synchronize()
        samples.append(1000 * (time.perf_counter() - start))
        print(f'{device:4s} 第 {i + 1:02d} 次：{samples[-1]:.3f} ms', flush=True)
    times[device] = statistics.median(samples)
    if device == 'cpu':
        reference = z
    else:
        torch.testing.assert_close(z.cpu(), reference, rtol=0.001, atol=0.001)
print(f'中位耗时：CPU {times['cpu']:.2f} ms / GPU {times['cuda']:.2f} ms', flush=True)
print(f'本次算子加速比：{times['cpu'] / times['cuda']:.2f}×；不代表完整训练加速比。', flush=True)
emit(cpu_ms=times['cpu'], gpu_ms=times['cuda'])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.barh(['CPU', 'GPU'], [times['cpu'], times['cuda']], color=['#94a3b8', '#10b981'])
ax.set_xlabel('Median matrix multiplication time (ms); lower is better')
ax.set_title('Your measured result')
plt.tight_layout()
plt.show()

## 3 · verl 负责什么？先理解奖励

大语言模型强化学习的基本循环是：**输入问题 → 模型生成回答（rollout）→ 给回答打分（reward）→ 更新策略（actor）→ 再生成**。

verl 负责组织这个训练流程；vLLM 常用于生成回答，PyTorch 用于计算梯度和更新参数，Ray 用于协调工作进程。

GRPO 会为同一个问题生成一组回答，用组内奖励比较来估计优势。下面先创建真实的 `verl.DataProto` 数据容器，再用四个回答展示简化的组内优势。它不是完整 GRPO 实现。

**观察：** 高于组内平均奖励的回答有正优势。如果所有回答奖励相同，组内比较就无法区分优劣。

In [ ]:
import torch
from verl import DataProto
batch = DataProto.from_dict(tensors={'input_ids': torch.tensor([[1, 2], [3, 4]])})
print(f'真实 verl.DataProto 批次：{len(batch)} 条数据', flush=True)
answers = ['4', '5', '4', '3']
scores = torch.tensor([float(x == '4') for x in answers])
advantages = (scores - scores.mean()) / (scores.std() + 1e-06)
for answer, score, advantage in zip(answers, scores, advantages):
    print(f'问题 2+2=? | 回答 {answer} | 奖励 {score:.0f} | 组内标准化优势 {advantage:+.3f}', flush=True)
print('优势 > 0：提高该回答概率；优势 < 0：降低该回答概率。', flush=True)
print('这是组内优势的简化示意；完整 GRPO 还涉及 token 掩码、策略比率与裁剪等。', flush=True)

## 4 · 看策略在 GPU 上一点点学会选择

现在执行一个小型 **REINFORCE 策略梯度实验**。对固定问题 `2+2=?`，策略从 `[3,4,5,6]` 采样；答对奖励为 1，否则为 0。用 `loss = -log_prob × reward` 计算损失、反向传播并更新参数。

这里只训练四个可学习分数，**没有训练语言模型，也没有调用 verl 训练器**。它用于理解奖励如何影响参数，且无需下载模型。与完整 PPO/GRPO 相比，它省略了策略裁剪、KL 约束、序列和分布式训练。

下方曲线会随计算实时更新。绿色表示答对的策略概率，灰色表示当前采样批次的实际奖励；采样奖励会波动。

In [ ]:
history = []
chart = display('', display_id=True)

def emit(**values):
    history.append(values)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot([v['step'] for v in history], [v['probability'] for v in history], color='#059669', label='P(correct)')
    ax.plot([v['step'] for v in history], [v['reward'] for v in history], color='#94a3b8', alpha=.7, label='Sample reward')
    ax.set(xlabel='Training step', ylabel='Probability / reward', ylim=(0, 1.05), title='Live GPU policy learning')
    ax.legend(loc='lower right')
    ax.grid(alpha=.15)
    fig.tight_layout()
    chart.update(fig)
    plt.close(fig)

import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
logits = torch.nn.Parameter(torch.zeros(4, device='cuda'))
optimizer = torch.optim.Adam([logits], lr=0.08)
print('GPU 上的四选一策略：2+2=?，候选答案 [3,4,5,6]。', flush=True)
print('从 25% 正确概率开始。使用 REINFORCE；这不是 verl/PPO/GRPO 完整训练。', flush=True)
for step in range(81):
    dist = torch.distributions.Categorical(logits=logits)
    probability = dist.probs[1].item()
    actions = dist.sample((128,))
    rewards = (actions == 1).float()
    if step % 2 == 0:
        emit(step=step, probability=probability, reward=rewards.mean().item())
        print(f'迭代 {step:02d} | 正确答案概率 {probability:.1%} | 本批平均奖励 {rewards.mean().item():.3f}', flush=True)
        time.sleep(0.08)
    if step == 80:
        break
    loss = -(dist.log_prob(actions) * rewards).mean()
    optimizer.zero_grad()
    loss.backward()
    assert torch.isfinite(logits.grad).all()
    optimizer.step()
assert probability > 0.85, '本次策略未达到预期，请检查日志'
print('PASS：真实采样 → 奖励 → 反向传播 → GPU 参数更新。', flush=True)
print('这里只学会了一个固定问题，不表示模型获得了通用数学能力。', flush=True)

### 小练习

1. 把学习率 `lr=0.08` 改成 `0.01`，重新运行第 4 节的代码格，观察相同步数内学习速度的变化。
2. 把 `rewards = (actions == 1).float()` 改成奖励第 0 个选项，会发生什么？同时修改显示标签和最终断言，让它们检查新的目标。
3. 想一想：如果奖励函数把错误答案评为高分，策略会学到什么？

每次重跑该格都会重新初始化策略。曲线变好只表示学会了这个固定任务，不能证明泛化能力。

## 5 · 加载真正的 verl 配置

下一格调用真实的 `verl.trainer.main_ppo`，使用 `--cfg job` **仅打印配置**，不会启动完整训练。

| 配置 | 作用 |
|---|---|
| `algorithm.adv_estimator=grpo` | 选择 GRPO 优势估计 |
| `actor_rollout_ref.model.path` | 初始模型目录或模型名 |
| `data.train_files` | 训练 Parquet 文件 |
| `actor_rollout_ref.rollout.n` | 每个提示生成的回答数；组内比较通常需要大于 1 |
| `actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu` | 每张 GPU 单次处理的小批量大小 |
| `trainer.n_gpus_per_node=1` | 每台机器使用一张 GPU |

**观察：** 配置打印成功代表训练入口可用，不代表某个具体模型已经完成训练。

In [ ]:
import subprocess
print('加载当前安装的 verl 训练配置，不启动大模型训练……', flush=True)
process = subprocess.Popen([sys.executable, '-u', '-m', 'verl.trainer.main_ppo', '--cfg', 'job', 'algorithm.adv_estimator=grpo', 'trainer.n_gpus_per_node=1', 'trainer.nnodes=1'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('verl 配置加载失败，请查看上方日志')
finally:
    if process.poll() is None:
        process.terminate()
    process.wait()
print('PASS：配置加载完成；配置成功不代表完整训练已验证。', flush=True)

## 6 · 从这个实验走向真实训练

真实训练还需要：一个预训练语言模型、符合格式的数据、能表达任务目标的奖励函数，以及适配显存的训练参数。本机仓库的 `local_smoke/` 有一个随机小 Qwen2 的试运行脚本；随机模型只适合检查流程，不适合作为问答能力示例。本教程不把它标为已通过的完整训练。

本教程所用版本的官方 GSM8K 快速入门要求 **至少 24GB GPU 显存**。使用 8GB 显卡时，需要调整批量和序列长度；需要另外验证短序列、小批量、LoRA 或卸载等具体配置。小模型也不能仅凭参数量保证整个训练流程放得下。

参考资料（网页可能随版本更新，本教程依据本机 verl 0.7.1）：

- [verl 官方文档](https://verl.readthedocs.io/en/latest/)
- [官方 GSM8K 快速入门](https://verl.readthedocs.io/en/latest/start/quickstart.html)
- [JupyterLab Notebook 操作](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)
- 官方 verl 源码中的 `docs/start/quickstart.rst`，以及本教程提供的 `local_smoke/run.sh`

### 常见问题

- **找不到 torch / verl**：切换到 `Python (verl GPU)` 内核。
- **长时间 `[*]`**：首次导入可能要十几秒；查看下方输出，需要时用 ■ 中断。
- **显存不足**：先重启内核释放本教程占用，关闭其他 GPU 计算，再降低实验规模。
- **网页断开**：保持启动 Notebook 的终端运行，重新打开终端打印的本机链接。
- **曲线是旧的**：保存后的文件可能显示上次结果；重跑第 4 节即可生成本次曲线。

完成后，你应该能解释：代码怎样进入 GPU、奖励怎样改变策略，以及 verl 如何把这些步骤组织成语言模型训练流程。